In [1]:
# pip install --upgrade transformers

In [2]:
import os
import json
import random
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset
import torch
from transformers import ViTFeatureExtractor, ViTForImageClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np
# from evaluate import load

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cpu


In [5]:
json_path = "/content/drive/MyDrive/Colab/Multimodal/Data/3k_image/1k_image_human.json"
img_dir = "/content/drive/MyDrive/Colab/Multimodal/Data/3k_image"

data for GOLD

In [6]:
# with open(json_path, 'r') as f:
#     data = json.load(f)

# label2idx = {'Non-sarcasm': 0, 'Sarcasm': 1}
# all_samples = [(item['image'], label2idx[item['label']]) for item in data]
# images, labels = zip(*all_samples)

# # Train (60%) + temp (40%)
# train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(images, labels, test_size=0.4, stratify=labels, random_state=2)

# # Dev (20%) + Test (20%) from temp
# val_imgs, test_imgs, val_labels, test_labels = train_test_split(temp_imgs, temp_labels, test_size=0.5, stratify=temp_labels, random_state=2)

# # Repack into (image, label) tuples
# train_data = list(zip(train_imgs, train_labels))
# val_data = list(zip(val_imgs, val_labels))
# test_data = list(zip(test_imgs, test_labels))

data for SILVER

In [7]:
base_img_dir = "/content/drive/MyDrive/Colab/Multimodal/Data/3k_image"
img_json_gold = "/content/drive/MyDrive/Colab/Multimodal/Data/GOLD/Image modality"
img_json_3k = "/content/drive/MyDrive/Colab/Multimodal/Data/Json"
train_json = os.path.join(img_json_3k, "image_full.json")
val_json   = os.path.join(img_json_gold, "dev_gold.json")
test_json  = os.path.join(img_json_gold, "test_gold.json")
ckpt_dir   = "/content/drive/MyDrive/Colab/Multimodal/Image/Pretrained_ViT/sandv"
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path  = os.path.join(ckpt_dir, "best_ViT_model_sandv.pth")

data for SILVER+GOLD

In [8]:
# idx2label = {0: 'Non-sarcasm', 1: 'Sarcasm'}

# # Save function
# def save_json(data_tuples, file_path):
#     json_data = [{"image": fname, "label": idx2label[label]} for fname, label in data_tuples]
#     with open(file_path, 'w') as f:
#         json.dump(json_data, f, indent=2)

# # Processing
# train_json_path = os.path.join(img_dir, "train_gold.json")
# val_json_path   = os.path.join(img_dir, "dev_gold.json")
# test_json_path  = os.path.join(img_dir, "test_gold.json")

# # Saving
# save_json(train_data, train_json_path)
# save_json(val_data, val_json_path)
# save_json(test_data, test_json_path)

In [9]:
class CustomImageDataset(Dataset):
    def __init__(self, json_file, img_dir, transform=None):
        with open(json_file, 'r') as f:
            self.data = json.load(f)
        self.img_dir = img_dir
        self.transform = transform
        self.label2idx = {'Non-sarcasm': 0, 'Sarcasm': 1}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = os.path.join(self.img_dir, item['image'])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image_tensor = self.transform(image)
        else:
            image_tensor = transforms.ToTensor()(image)

        label = self.label2idx[item['label']]
        return {'pixel_values': image_tensor, 'labels': label}

Feature Extractor

In [10]:
model_id = "google/vit-base-patch16-224-in21k"
feature_extractor = ViTFeatureExtractor.from_pretrained(model_id)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [11]:
def transform_fn(image):
    encoding = feature_extractor(image, return_tensors='pt')
    return encoding['pixel_values'].squeeze()

In [12]:
train_dataset = CustomImageDataset(train_json, img_dir, transform=transform_fn)
val_dataset = CustomImageDataset(val_json, img_dir, transform=transform_fn)
test_dataset = CustomImageDataset(test_json, img_dir, transform=transform_fn)

In [13]:
def collate_fn(batch):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = torch.tensor([item['labels'] for item in batch])
    return {'pixel_values': pixel_values, 'labels': labels}

Load model

In [14]:
model = ViTForImageClassification.from_pretrained(
    model_id,
    num_labels=2,
    id2label={0: 'Non-sarcasm', 1: 'Sarcasm'},
    # label2id=label2idx,
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab/Multimodal/Image/Pretrained_ViT/sandv",
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    eval_strategy="steps",
    save_steps=1000,
    eval_steps=5,
    logging_steps=10,
    learning_rate=2e-4,
    num_train_epochs=4,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    precision = precision_score(labels, predictions, average='binary')
    recall = recall_score(labels, predictions, average='binary')

    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ply58509 (ply58509-uit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
test_results = trainer.evaluate(test_dataset)
print("Final test metrics:")
for k, v in test_results.items():
    print(f"{k}: {v:.4f}")